In [50]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import pandas as pd
from collections import Counter
from tqdm import tqdm
from itertools import combinations
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import numpy as np
import os
import ast

# Cargar Datos

In [51]:
file_path = r"/kaggle/input/corpus-para-ner/chunks_etiquetados_binario.xlsx"
chunks_etiquetados = pd.read_excel(file_path, engine='openpyxl')

file_path2 = r"/kaggle/input/corpus-para-ner/corpus_cleaned.xlsx"
corpus_cleaned = pd.read_excel(file_path2, engine='openpyxl')

chunks_df = pd.read_parquet(r"/kaggle/input/corpus-para-ner/chunks.parquet")

# Cargar modelos

In [52]:
tokenizer = AutoTokenizer.from_pretrained("Babelscape/wikineural-multilingual-ner")
model = AutoModelForTokenClassification.from_pretrained("Babelscape/wikineural-multilingual-ner")
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, grouped_entities=True)


Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/token_classification.py:181: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


In [45]:
tokenizer.model_max_length

512

In [53]:
def extraer_entidades(texto, max_palabras=300):
    """
    Extrae entidades nombradas de un texto largo usando el modelo NER.
    Divide automáticamente el texto en fragmentos (máx. 400 palabras)
    para evitar el límite de tokens del modelo.

    Devuelve: lista de tuplas (tipo_entidad, texto_entidad)
    """
    entidades = []
    try:
        palabras = texto.split()
        for i in range(0, len(palabras), max_palabras):
            fragmento = " ".join(palabras[i:i + max_palabras])
            resultados = ner_pipeline(fragmento)
            for r in resultados:
                tipo = r.get("entity_group", "").strip()
                nombre = r.get("word", "").strip()
                if tipo and nombre:
                    entidades.append((tipo, nombre))
    except Exception as e:
        print(f"⚠️ Error procesando texto: {e}")
    return entidades

# Extraer entidades general

In [54]:
chunks_etiquetados.head(2)

,chunk_id,id_doc,texto_chunk,Ciencias_ambientales_ingenieria,Ciencias_espacio,Ciencias_fisicas,Ciencias_Geografia_oceanografia,Ciencias_medicas,Ciencias_metereologia,Ciencias_naturales,...,Ciencias_tierra,Ciencia_Administracion_ciencia_investigacion,Ciencia_biologia,Ciencia_enfoque_cientifico,Ciencia_hidrologia,Ciencia_matematicas_estadistica,Ciencia_patologia,Ciencia_recursos_naturales,categorias_detectadas,etiqueta_ciencia
0,0,1,"La Coalición Colombia Partido Alianza Verde, P...",0.411230,0.368818,0.363447,0.354240,0.378154,0.343397,0.387251,...,0.392998,0.423331,0.337017,0.352550,0.328395,0.394268,0.393815,0.389038,"[('ninguna', 0)]",0
1,1,1,"al mismo tiempo lo exponen, en ciertas ocasion...",0.331604,0.304309,0.346682,0.315253,0.344347,0.323262,0.344607,...,0.318409,0.378231,0.317467,0.368125,0.306375,0.381964,0.337864,0.306510,"[('ninguna', 0)]",0


In [55]:
tqdm.pandas(desc="Extrayendo entidades")
chunks_etiquetados['entidades'] = chunks_etiquetados['texto_chunk'].progress_apply(extraer_entidades)

Extrayendo entidades: 100%|██████████| 62651/62651 [22:22<00:00, 46.67it/s]


In [59]:
resultado = chunks_etiquetados[["chunk_id", "id_doc", "texto_chunk", "categorias_detectadas", "etiqueta_ciencia","entidades"]]

resultado.to_parquet(r"/kaggle/working/general_ner.parquet", index=False)

# Leer NER Creado

In [60]:
ner_general = pd.read_parquet(r"/kaggle/working/general_ner.parquet")
ner_general

,chunk_id,id_doc,texto_chunk,categorias_detectadas,etiqueta_ciencia,entidades
0,0,1,"La Coalición Colombia Partido Alianza Verde, P...","[('ninguna', 0)]",0,"[[ORG, Coalición Colombia Partido Alianza Verd..."
1,1,1,"al mismo tiempo lo exponen, en ciertas ocasion...","[('ninguna', 0)]",0,"[[ORG, RCN Radio], [PER, Jorge Enrique Robledo..."
2,2,1,los acuerdos con las Farc. Anunció que no prom...,"[('ninguna', 0)]",0,"[[ORG, Far], [MISC, Dian], [PER, Fajardo]]"
3,3,1,moratoria en la explotación tipo fracking. Y f...,"[('ninguna', 0)]",0,"[[PER, Fajardo], [ORG, Pontificia Universidad ..."
4,0,2,Las interpretaciones de la historia sirven com...,"[('ninguna', 0)]",0,"[[MISC, G], [LOC, Balcanes]]"
...,...,...,...,...,...,...
62646,7,13676,"así, encuentra el 90 del alimento que guardó. ...","[('ninguna', 0)]",0,"[[PER, Piaget], [PER, Rodolfo Llinás]]"
62647,8,13676,capacidad de predecir un evento futuro. Por lo...,"[('Ciencias_naturales', 0.48311546444892883)]",1,"[[LOC, Escandinavia]]"
62648,9,13676,similar. Se ha encontrado que éstos le roban l...,"[('ninguna', 0)]",0,[]
62649,10,13676,está siendo parcialmente validada con experime...,"[('Ciencias_naturales', 0.5003376007080078)]",1,"[[PER, Brecht]]"


In [61]:
ner_general['entidades'][10]

array([array(['PER', 'Filipo'], dtype=object),
       array(['PER', 'Alejandro Magno'], dtype=object),
       array(['PER', 'Aristóteles'], dtype=object),
       array(['ORG', 'FYROM'], dtype=object),
       array(['PER', 'Zoran Zaev'], dtype=object),
       array(['LOC', 'Tesaloniki'], dtype=object),
       array(['PER', 'Yannis Butaris'], dtype=object),
       array(['LOC', 'Krusevo'], dtype=object),
       array(['PER', 'Zaev'], dtype=object),
       array(['LOC', 'Aeropuerto Macedonia'], dtype=object),
       array(['LOC', 'Tesaloniki'], dtype=object),
       array(['LOC', 'Macedonia'], dtype=object),
       array(['PER', 'Butaris'], dtype=object)], dtype=object)

# Analizar entidades

In [73]:
# Función para extraer entidades
def extract_entities(entities_array):
    """Extrae las entidades del formato array"""
    entities_list = []
    
    if isinstance(entities_array, str):
        try:
            entities_array = ast.literal_eval(entities_array)
        except:
            return entities_list
    
    if hasattr(entities_array, '__iter__'):
        for item in entities_array:
            if isinstance(item, (list, tuple, np.ndarray)) and len(item) >= 2:
                entity_type = str(item[0])
                entity_name = str(item[1])
                entities_list.append((entity_type, entity_name))
    
    return entities_list

In [75]:
# Extraer todas las entidades manteniendo la etiqueta de origen
entidades_con_etiqueta = []

for _, row in ner_general.iterrows():
    entidades = extract_entities(row['entidades'])
    etiqueta = row['etiqueta_ciencia']
    
    for entidad in entidades:
        entidades_con_etiqueta.append({
            'tipo': entidad[0],
            'nombre': entidad[1],
            'etiqueta_origen': etiqueta
        })

# Crear DataFrame temporal
df_temp = pd.DataFrame(entidades_con_etiqueta)

# Calcular frecuencias totales y por categoría
resultado = df_temp.groupby(['tipo', 'nombre']).agg(
    frecuencia_total=('etiqueta_origen', 'size'),
    frecuencia_ciencia=('etiqueta_origen', lambda x: (x == 1).sum()),
    etiqueta_ciencia=('etiqueta_origen', lambda x: 1 if any(x == 1) else 0)
).reset_index()

# Calcular frecuencia_no_ciencia y cociente
resultado['frecuencia_no_ciencia'] = resultado['frecuencia_total'] - resultado['frecuencia_ciencia']
resultado['cociente'] = resultado['frecuencia_ciencia'] / resultado['frecuencia_total']

# Ordenar columnas y renombrar
resultado = resultado[[
    'tipo', 'nombre', 'frecuencia_total', 
    'frecuencia_no_ciencia', 'frecuencia_ciencia',
    'cociente', 'etiqueta_ciencia'
]]

# Renombrar columnas
resultado.columns = ['Tipo', 'Entidad', 'Frecuencia_Total', 
                     'Frecuencia_No_Ciencia', 'Frecuencia_Ciencia',
                     'Cociente', 'Etiqueta_Ciencia']

# Ordenar por frecuencia total descendente
resultado = resultado.sort_values('Frecuencia_Total', ascending=False).reset_index(drop=True)

# Formatear cociente
resultado['Cociente'] = resultado['Cociente'].round(4)

# Guardar a Excel
resultado.to_excel('entidades_conteo.xlsx', index=False)

print(f"Proceso completado. Se encontraron {len(resultado)} entidades únicas.")
print(f"Archivo guardado: entidades_conteo.xlsx")
print("\nPrimeras 5 filas:")
print(resultado.head())

Proceso completado. Se encontraron 65026 entidades únicas.
Archivo guardado: entidades_conteo.xlsx

Primeras 5 filas:
  Tipo         Entidad  Frecuencia_Total  Frecuencia_No_Ciencia  \
0  LOC        Colombia             14815                  13443   
1  PER           Duque              6159                   5893   
2  LOC          Estado              5486                   4582   
3  LOC  Estados Unidos              4163                   3889   
4  PER           Uribe              4140                   4064   

   Frecuencia_Ciencia  Cociente  Etiqueta_Ciencia  
0                1372    0.0926                 1  
1                 266    0.0432                 1  
2                 904    0.1648                 1  
3                 274    0.0658                 1  
4                  76    0.0184                 1  
